In [ ]:
# ── Which chain this notebook reproduces ──────────────────────────────────────
#
# Both numbers are macro over the 199 out-of-fold stacks of
# edge_run/2026-09-13/0/preds_0.45 -- the run EDGE_W points at -- so they are
# the same nodes, the same edge model and the same post-processing, differing
# only in the gate.
#
#   False   main_edge_postprocess.py --ilp on the current DEFAULTS block
#           cv 0.8735    adj_edge_jaccard 0.8574    division_jaccard 0.1619
#
#   True    the same chain, then division_cls/apply_fork_gate.py deletes every
#           fork the division classifier scores below DIVISION_TAU
#           cv 0.8791    adj_edge_jaccard 0.8576    division_jaccard 0.2150
#
# The gate only ever deletes second-child edges, so adj_edge_jaccard is
# unchanged (+0.0002) and the whole +0.0056 is the division term: it removes
# 121 of 138 division false positives and 9 of 46 true ones. It costs one extra
# GPU pass, over 32x32 fork crops rather than the whole volume.
#
# Two caveats on the True branch, both from running on a test split:
#
#  * The 0.8791 was measured leave-one-embryo-out, each embryo scored by the
#    model that held it out. A test stack belongs to neither fold, so the gate
#    averages both fold models. The two differ in strength (AP 0.627 / 0.426),
#    so this is the closest available estimator, not the same one.
#  * DIVISION_TAU is not tuned: the classifier saturates (positives median
#    1.000, negatives 0.000) and the result is unchanged from 0.1 to 0.7.
USE_DIVISION_CLS = True
DIVISION_TAU = 0.5

## Setup

In [1]:
import os
import json
import numpy as np
import pandas as pd

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [2]:
!cp -r /kaggle/input/datasets/theoviel/biohub-cell-tracking-code/src ./

In [3]:
try:
    import zarr, geff, tracksdata
    
except: 
    import os
    import sys
    import subprocess
    import importlib
    import importlib.util
    from pathlib import Path
    
    os.environ.setdefault("POLARS_PREFER_PKG", "32")
    
    SUPPORT_DIR = Path(
        "/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1"
    )
    WHEELS_DIR = SUPPORT_DIR / "wheels"
    
    if not WHEELS_DIR.exists():
        candidates = list(Path("/kaggle/input").glob("**/wheels"))
        if not candidates:
            raise FileNotFoundError("Could not find the attached offline wheels directory")
        WHEELS_DIR = candidates[0]
    
    print("Offline wheels:", WHEELS_DIR)
    
    OFFLINE_PACKAGES = [
        "tracksdata",
        "zarr==3.2.1",
        "numcodecs==0.15.1",
        "donfig==0.8.1.post1",
        "geff==1.2.0.1.1",
        "geff-spec==1.1.1",
        "pyscipopt==6.2.1",
        "ilpy==0.6.0",
        "rustworkx==0.18.0",
        "polars==1.42.0",
        "polars-runtime-32==1.42.0",
        "bidict==0.23.1",
        "imagecodecs==2026.6.26",
    ]
    
    def module_missing(module_name: str) -> bool:
        return importlib.util.find_spec(module_name) is None
    
    
    REQUIRED_IMPORTS = {
        "tracksdata": "tracksdata",
        "zarr": "zarr",
        "numcodecs": "numcodecs",
        "geff": "geff",
        "pyscipopt": "pyscipopt",
        "ilpy": "ilpy",
        "rustworkx": "rustworkx",
        "polars": "polars",
        "imagecodecs": "imagecodecs",
    }
    
    
    def purge_modules(module_roots):
        """
        Remove already-imported package modules from sys.modules.
    
        Normally this cell runs before imports, but this also protects against
        accidental imports performed by earlier Kaggle initialization code.
        """
        for root in module_roots:
            for name in list(sys.modules):
                if name == root or name.startswith(root + "."):
                    sys.modules.pop(name, None)
    
    
    def install_offline_packages():
        cmd = [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--no-index",
            "--no-deps",
            "--find-links",
            str(WHEELS_DIR),
            *OFFLINE_PACKAGES,
        ]
    
        print("Installing attached packages without modifying NumPy/SciPy/Torch...")
        result = subprocess.run(
            cmd,
            text=True,
            capture_output=True,
        )
    
        if result.returncode != 0:
            print(result.stdout[-4000:])
            print(result.stderr[-4000:])
            raise RuntimeError("Offline dependency installation failed")
    
        purge_modules(REQUIRED_IMPORTS.values())
    
    
    install_offline_packages()
    
    
    failures = {}
    
    for name, module_name in {
        **REQUIRED_IMPORTS,
        "numpy": "numpy",
        "scipy": "scipy",
        "dask": "dask.array",
        "xarray": "xarray",
    }.items():
        try:
            importlib.import_module(module_name)
        except Exception as exc:
            failures[name] = f"{type(exc).__name__}: {exc}"
    
    if failures:
        raise ImportError(
            "Dependency verification failed:\n"
            + "\n".join(f"{name}: {error}" for name, error in failures.items())
        )

import zarr, geff, tracksdata
print("All offline dependencies imported successfully.")

Offline wheels: /kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1/wheels
Installing attached packages without modifying NumPy/SciPy/Torch...
All offline dependencies imported successfully.


/usr/local/lib/python3.12/dist-packages/dask/array/image.py:7: FutureWarning: `find_available_plugins` is deprecated since version 0.25 and will be removed in version 0.27. The plugin infrastructure of `skimage.io` is deprecated. Instead, use `imageio` or other I/O packages directly.
  from skimage.io import imread as sk_imread
/usr/lib/python3.12/importlib/__init__.py:90: FutureWarning: `reset_plugins` is deprecated since version 0.25 and will be removed in version 0.27. The plugin infrastructure of `skimage.io` is deprecated. Instead, use `imageio` or other I/O packages directly.
  return _bootstrap._gcd_import(name[level:], package, level)


## Configuration

In [ ]:
# Weights
TRACK_W = "/kaggle/input/datasets/theoviel/biohub-weights-2/tracking_2026-09-13_1/"
CONTRASTIVE_W = "/kaggle/input/datasets/theoviel/biohub-weights-2/contrastive_2026-09-09_1/"
EDGE_W = "/kaggle/input/datasets/theoviel/biohub-weights-2/edge_2026-09-13_0/"
# fold_0/ and fold_1/ ckpt_final.pt from logs/division/2026-09-16/0. Only read
# when USE_DIVISION_CLS; the assert is here rather than at the gate so a missing
# upload fails in the configuration cell instead of after an hour of inference.
DIVISION_W = "/kaggle/input/datasets/theoviel/biohub-weights-2/division_2026-09-16_0/"

edge_args = json.load(open(EDGE_W + "args.json"))
EMB_KEY = edge_args.get("embeddings_key") or "embeddings"
assert EMB_KEY.replace('embeddings_', '') in CONTRASTIVE_W, f'EMB_KEY "{EMB_KEY}" does not match CONTRASTIVE_W'

if USE_DIVISION_CLS:
    import glob
    _ckpts = sorted(glob.glob(DIVISION_W + "fold_*/ckpt_final.pt"))
    assert _ckpts, f"USE_DIVISION_CLS is set but no fold_*/ckpt_final.pt under {DIVISION_W}"
    print(f"-> division classifier: {len(_ckpts)} fold models, tau {DIVISION_TAU}")

# Params
DET_THRESHOLD = 0.40
NMS_RADIUS_UM = 6.0
EDGE_THRESHOLD = 0.45
MAX_NODES = 1500

# Data
DATA_DIR = "/kaggle/input/competitions/biohub-cell-tracking-during-development/test/"

if len(os.listdir(DATA_DIR)) == 4:
    !mkdir -p /tmp/data/
    !cp -r /kaggle/input/competitions/biohub-cell-tracking-during-development/train/44b6_18ced818.zarr /tmp/data/
    DATA_DIR = "/tmp/data/"

## 1. Detection and tracking

The U-Net detector plus the node transformer, as before. Two things differ from
the previous submission: the checkpoint carries the extra decoder stage (the
heatmap comes out at 1/2 Y/X while the node features stay on the isotropic 1/4
grid), and the NMS radius is passed explicitly rather than inherited from
`args.json` — see the note in the configuration cell.

Output is the node set every later stage reads.

In [5]:
%%writefile inference.py

import sys
sys.path.append('./src/')

import warnings
warnings.filterwarnings("ignore")

import os
import json
import torch
import argparse
import numpy as np
from pathlib import Path

from tracking.main import build_model
from tracking_cellmot.io import list_datasets
from tracking.predict import PredictConfig, predict_video_thresholds
from training.distributed import cleanup_distributed, init_distributed
from utils.logger import Config

DATA_DIR = "/kaggle/input/competitions/biohub-cell-tracking-during-development/test/"


def parse_args():
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--log_folder", required=True, help="run folder holding args.json and the checkpoint")
    p.add_argument("--ckpt", default="fold_fullfit/ckpt_final.pt",
                   help="checkpoint, relative to --log_folder; searched for if not found there")
    p.add_argument("--det_threshold", type=float, default=0.40,
                   help="detection threshold; pick it from the run's cv_score")
    p.add_argument("--nms_radius_um", type=float, default=None,
                   help="greedy-NMS radius. Overrides the training run's own "
                        "value, which is a training default and not the one the "
                        "downstream node set was chosen at -- preds_r6_0.40 "
                        "means radius 6, while args.json records 7")
    p.add_argument("--out_dir", default=None, help="default <log_folder>/preds")
    p.add_argument("--limit", type=int, default=None, help="cap the stacks (debugging)")
    p.add_argument("--no_fp16", action="store_true",
                   help="run the detector in fp32. Measured on an H100, where "
                        "fp32 convs are really TF32 so the gap is at its "
                        "narrowest: the UNet forward is 54% of this stage and "
                        "fp16 takes it 3.34s -> 2.53s per 30 frames (-24%), "
                        "total -12%. A T4/P100 has no TF32, so expect more "
                        "there. It moves detections: 6682 nodes against 6683 "
                        "on one stack, threshold crossings in the last digit")
    p.add_argument("--data_dir", default=DATA_DIR, help="override the test split")
    return p.parse_args()


def resolve_ckpt(log_folder: Path, ckpt: str) -> Path:
    """The checkpoint, wherever the weights dataset happens to have put it.

    A run folder keeps checkpoints under fold_*/; a weights dataset is often
    flattened to the root instead. Rather than making the caller know which,
    take the path as given and fall back to a search by file name.
    """
    direct = log_folder / ckpt
    if direct.exists():
        return direct
    found = sorted(log_folder.rglob(Path(ckpt).name))
    if not found:
        available = sorted(str(f.relative_to(log_folder)) for f in log_folder.rglob("*.pt"))
        raise FileNotFoundError(
            f"{ckpt} not found under {log_folder}. Checkpoints present: {available}"
        )
    return found[0]


def main():
    args = parse_args()
    ctx = init_distributed(None)
    device = ctx.device

    # Every window is the same shape, so cuDNN's autotuner pays for itself on
    # the first few and wins on the rest. Worth ~0.4s per 30 frames here on top
    # of fp16; on a fixed-shape 3D conv stack it is free money.
    torch.backends.cudnn.benchmark = True

    paths = sorted(list_datasets(Path(args.data_dir), require_geff=False))
    if args.limit:
        paths = paths[: args.limit]

    log_folder = Path(args.log_folder)
    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    ckpt_path = resolve_ckpt(log_folder, args.ckpt)

    cfg = Config(json.load(open(log_folder / "args.json")))

    # Inference settings come from the training run wherever the run fixes
    # them, so the node sets the edge head sees here are the ones it was fitted
    # against. Only the detection threshold is chosen separately -- it is swept
    # after training, and the value below is the sweep's answer.
    pred_cfg = PredictConfig
    pred_cfg.det_threshold = args.det_threshold
    pred_cfg.normalization = getattr(cfg, "normalization", None)
    pred_cfg.nms = getattr(cfg, "nms", pred_cfg.nms)
    pred_cfg.nms_radius_um = (
        args.nms_radius_um if args.nms_radius_um is not None
        else getattr(cfg, "nms_radius_um", pred_cfg.nms_radius_um)
    )
    pred_cfg.pool_kernel_um = getattr(cfg, "pool_kernel_um", pred_cfg.pool_kernel_um)
    pred_cfg.use_offset = bool(getattr(cfg, "offset_head", False))

    window_size = getattr(cfg, "window_size", 2)
    downscale_factor = getattr(cfg, "downscale_factor", 1)

    if ctx.is_main:
        print(f"-> log_folder     : {log_folder}")
        print(f"-> checkpoint     : {ckpt_path.relative_to(log_folder)}")
        print(f"-> threshold      : {pred_cfg.det_threshold}")
        print(f"-> nms            : {pred_cfg.nms} @ {pred_cfg.nms_radius_um} um")
        print(f"-> normalization  : {pred_cfg.normalization}")
        print(f"-> window_size    : {window_size}")
        print(f"-> stacks         : {len(paths)}")
        print(f"-> writing to     : {out_dir}")
        if ctx.distributed:
            print(f"-> distributed    : {ctx.world_size} processes")
        print()

    state = torch.load(ckpt_path, map_location="cpu")
    state = state.get("model_state_dict", state)
    state = {k.replace("unet.module.", "unet.", 1): v for k, v in state.items()}

    model = build_model(cfg, None)
    model.load_state_dict(state, strict=True)
    model.eval().to(device)

    # Shard stacks across ranks; each writes its own npz files.
    mine = paths[ctx.rank :: ctx.world_size]

    for i, path in enumerate(mine):
        print(f"[rank {ctx.rank}] {i + 1}/{len(mine)}  -  {path.name}", flush=True)

        # predict_video_thresholds rather than predict_video: the latter drops
        # the per-node detection probability, and the edge model reads it as a
        # feature (`use_conf_score`). Without a `node_probs` array its dataset
        # refuses to build, so the pipeline would fail two stages later.
        # autocast here rather than inside predict_video_thresholds: that
        # function is also the training-time evaluator, and its numbers are
        # compared against training losses computed in fp32.
        with torch.autocast("cuda", torch.float16,
                            enabled=not args.no_fp16 and device.type == "cuda"):
            results, _ = predict_video_thresholds(
                model,
                path,
                thresholds=[pred_cfg.det_threshold],
                device=device,
                cfg=pred_cfg,
                window_size=window_size,
                downscale_factor=downscale_factor,
                progress=ctx.is_main,
                return_det_logits=False,
                nms_radii=[pred_cfg.nms_radius_um],
            )
        # Keyed (head, radius, threshold), five values. "det" is the plain
        # detection head; this detector also carries an annotation head, and
        # the node set the edge model was fitted on came from "det", so asking
        # for the other one here would score its candidates against a node set
        # it never saw. The trailing two -- the annotation probability and the
        # unrounded coordinates -- are for consumers that still have fusing to
        # do, which this pipeline does not.
        coords, edges, node_probs, _annot, _coords_f = results[
            ("det", float(pred_cfg.nms_radius_um), float(pred_cfg.det_threshold))
        ]

        np.savez_compressed(
            out_dir / f"{path.name}.npz",
            coords=coords,
            edges=np.asarray(edges, dtype=np.float64).reshape(-1, 4),
            # Row-aligned with coords, exactly as main_predict.py writes it.
            node_probs=np.asarray(node_probs, dtype=np.float32).reshape(-1),
        )


if __name__ == "__main__":
    try:
        main()
    finally:
        cleanup_distributed()


Writing inference.py


In [6]:
# Detection settings are forced rather than inherited: NMS_RADIUS_UM is the
# radius the downstream node set was chosen at, which is not the one in the
# tracking run's args.json.
cmd = (
    "torchrun --standalone --nproc_per_node=2 inference.py"
    f" --data_dir '{DATA_DIR}'"
    f" --log_folder '{TRACK_W}'"
    " --ckpt 'fold_fullfit/ckpt_final.pt'"
    f" --det_threshold {DET_THRESHOLD}"
    f" --nms_radius_um {NMS_RADIUS_UM}"
    " --out_dir /tmp/preds/"
)
print(cmd, "\n")
!{cmd}


torchrun --standalone --nproc_per_node=2 inference.py --data_dir '/tmp/data/' --log_folder '/kaggle/input/datasets/theoviel/biohub-weights-2/tracking_2026-09-09_4/' --ckpt 'fold_fullfit/ckpt_final.pt' --det_threshold 0.4 --nms_radius_um 6.0 --out_dir /tmp/preds/ 

W0910 11:37:48.338000 68 torch/distributed/run.py:852] 
W0910 11:37:48.338000 68 torch/distributed/run.py:852] *****************************************
W0910 11:37:48.338000 68 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0910 11:37:48.338000 68 torch/distributed/run.py:852] *****************************************
[W910 11:37:48.164505825 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W910 11:37:55.530254503 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


## 2. Contrastive embeddings (NETr)

One appearance vector per detection, written **back into the detection npz**
rather than into a folder of its own, so the coordinates and their embeddings
cannot drift apart. The edge model reads them as node features; without them it
falls back to geometry alone and scores materially worse.

In [7]:
# NETr embeddings for every detection, written back into /tmp/preds/*.npz under
# EMB_KEY. --fold fullfit runs the one model over every stack: the out-of-fold
# default has no meaning on a test split whose embryos are not the training ones.
cmd = (
    "torchrun --standalone --nproc_per_node=2 src/inference_contrastive.py"
    f" --exp_folder '{CONTRASTIVE_W}'"
    " --coords_log_folder /tmp/preds/"
    f" --data_dir '{DATA_DIR}'"
    " --fold fullfit"
    f" --key '{EMB_KEY}'"
)
print(cmd, "\n")
!{cmd}


torchrun --standalone --nproc_per_node=2 src/inference_contrastive.py --exp_folder '/kaggle/input/datasets/theoviel/biohub-weights-2/contrastive_2026-09-09_1/' --coords_log_folder /tmp/preds/ --data_dir '/tmp/data/' --fold fullfit --key 'embeddings_contrastive_2026-09-09_1' 

W0910 11:40:43.766000 125 torch/distributed/run.py:852] 
W0910 11:40:43.766000 125 torch/distributed/run.py:852] *****************************************
W0910 11:40:43.766000 125 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0910 11:40:43.766000 125 torch/distributed/run.py:852] *****************************************
[W910 11:40:44.512935911 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W910 11:40:47.102199563 socket.cpp:207] [c10d] The hostname of the client socket cannot be re

## 3. Edge transformer

Scores every candidate link, and saves **all** of them alongside the committed
ones. That is what makes the threshold and the linking rule a post-processing
decision instead of a GPU one — the next cell re-solves the assignment from this
array, so the threshold here does not reach the final graph.

In [8]:
%%writefile edge_inference.py
"""Score every candidate link with the edge transformer, on the test split.

`main_edge_predict.py` cannot be used as-is here: it picks its videos with
`load_videos(..., require_geff=True)` and slices them by fold and embryo, and
the test split has neither ground truth nor known embryo names. Everything below
the video list is the same library code that produced the out-of-fold numbers --
`load_video`, `dataset_for`, `score_video`, `link_video` -- so the only thing
this file changes is which stacks are fed in and which checkpoint reads them.

Writes coords + committed edges + the full scored candidate array, which is what
`main_edge_postprocess.py` re-links from.
"""

import sys
sys.path.append('./src/')

import warnings
warnings.filterwarnings("ignore")

import argparse
import json
from pathlib import Path

import dataclasses

import numpy as np
import torch

from edge_transformer.data import load_video
from edge_transformer.infer import dataset_for, load_hoct
from edge_transformer import predict as edge_predict
from edge_transformer.predict import link_video, score_video
from embeddings.dataset import find_coord_files
from tracking_cellmot.io import list_datasets
from training.distributed import cleanup_distributed, init_distributed


def parse_args():
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument("--log_folder", required=True, help="edge run: args.json + checkpoint")
    p.add_argument("--ckpt", default="fold_fullfit/ckpt_final.pt")
    p.add_argument("--nodes_folder", required=True, help="detections, with embeddings")
    p.add_argument("--data_dir", required=True)
    p.add_argument("--out_dir", required=True)
    p.add_argument("--threshold", type=float, default=0.45)
    p.add_argument("--embeddings_key", default=None)
    p.add_argument("--max_nodes_per_frame", type=int, default=None,
                   help="EdgeDataset's own cap, which drops nodes at RANDOM and "
                        "re-draws per window. A training-memory knob; use "
                        "--max_nodes instead")
    p.add_argument("--edge_pair_budget", type=int, default=0,
                   help="cap on B * E_max^2 per forward, 0 = the library default "
                        "(32M). This, not --max_nodes, is what actually governs "
                        "peak GPU memory here -- halving it roughly halves the "
                        "peak, at some wall-clock cost")
    p.add_argument("--max_nodes", type=int, default=0,
                   help="keep only the N most confident detections per frame, "
                        "0 = no cap. A safety net against OOM on an unusually "
                        "crowded stack: the edge model's cost is quadratic in "
                        "the nodes of a frame pair")
    p.add_argument("--no_fp16", action="store_true")
    p.add_argument("--limit", type=int, default=None)
    return p.parse_args()


def resolve_ckpt(log_folder: Path, ckpt: str) -> Path:
    """The checkpoint wherever the weights dataset put it -- see inference.py."""
    direct = log_folder / ckpt
    if direct.exists():
        return direct
    found = sorted(log_folder.rglob(Path(ckpt).name))
    if not found:
        available = sorted(str(f.relative_to(log_folder)) for f in log_folder.rglob("*.pt"))
        raise FileNotFoundError(f"{ckpt} not found under {log_folder}. Present: {available}")
    return found[0]


def cap_nodes(video, max_nodes: int):
    """Keep the ``max_nodes`` most confident detections of each frame.

    A bound on the edge model's cost, which is quadratic in the nodes of a frame
    pair. Two things make this different from ``EdgeDataset``'s own
    ``max_nodes_per_frame``, and both matter for inference:

    * **It ranks by confidence.** The dataset's cap drops at random, which is
      right for training -- the survivors stay a fair sample -- and wrong here,
      where a dropped node gets no candidate edges at all and simply cannot be
      linked. If something has to go, it should be the detections the model was
      least sure of.
    * **It is applied once, to the node set.** The dataset's cap is re-drawn per
      window, so a node can be present in one window and absent from the next,
      and the graph a video produces is not a graph of any one node set.

    A cap that never binds costs nothing, so the default is off: the reported
    0.8699 is an uncapped number, and turning this on is a decision to trade
    recall for a memory ceiling.
    """
    if not max_nodes:
        return video
    if video.conf is None:
        raise ValueError(
            "--max_nodes ranks by detection confidence, but this node set has "
            "no 'node_probs' array. Re-run the detection stage, or drop the cap."
        )

    keep_parts = []
    for t in sorted(video.rows_by_frame):
        rows = video.rows_by_frame[t]
        if len(rows) <= max_nodes:
            keep_parts.append(rows)
            continue
        # stable, so ties fall back to file order rather than to whatever the
        # sort happens to do -- and keep ascending, since every consumer reads a
        # frame's rows in file order.
        best = np.argsort(-video.conf[rows], kind="stable")[:max_nodes]
        keep_parts.append(np.sort(rows[best]))
    keep = np.concatenate(keep_parts) if keep_parts else np.empty(0, int)

    if len(keep) == len(video.coords):
        return video

    coords = video.coords[keep]
    frame_of = coords[:, 0].astype(int)

    # Every surviving row moves, so anything indexing into coords has to move
    # with it. Ground-truth links do on the training split -- inference has
    # none, but refusing to handle them would make this untestable against the
    # only data that can say what the cap costs.
    remap = np.full(len(video.coords), -1, dtype=np.int64)
    remap[keep] = np.arange(len(keep))
    links = {}
    for t, arr in (video.links or {}).items():
        arr = np.asarray(arr)
        if not len(arr):
            links[t] = arr
            continue
        src, tgt = remap[arr[:, 0].astype(int)], remap[arr[:, 1].astype(int)]
        ok = (src >= 0) & (tgt >= 0)          # a link whose endpoint was cut is gone
        out = arr[ok].copy()
        out[:, 0], out[:, 1] = src[ok], tgt[ok]
        links[t] = out

    return dataclasses.replace(
        video,
        coords=coords,
        emb=None if video.emb is None else video.emb[keep],
        conf=video.conf[keep],
        rows_by_frame={int(t): np.flatnonzero(frame_of == t) for t in np.unique(frame_of)},
        links=links,
    )


def candidate_array(video, scored):
    """(src row, tgt row, prob, distance um, dt) -- a copy of main_edge_predict's.

    Kept identical on purpose: it is the post-processing chain's entire input,
    and the out-of-fold tuning was done against exactly this layout.
    """
    frame_of = np.asarray(video.coords)[:, 0].astype(int)
    if not scored:
        return np.empty((0, 5))
    return np.asarray(
        [(a, b, p, d, frame_of[b] - frame_of[a]) for (a, b), (p, d) in scored.items()],
        dtype=np.float64,
    ).reshape(-1, 5)


def main():
    args = parse_args()
    ctx = init_distributed(None)
    device = ctx.device

    if args.edge_pair_budget:
        # A module constant read inside score_video's batch planner, so it is set
        # rather than passed. Done before any scoring, and echoed below, because
        # a memory knob that is not in the log is a memory knob nobody can
        # reproduce.
        edge_predict.EDGE_PAIR_BUDGET = int(args.edge_pair_budget)

    log_folder = Path(args.log_folder)
    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    ckpt_path = resolve_ckpt(log_folder, args.ckpt)

    model, cfg = load_hoct(str(ckpt_path), device=device, verbose=ctx.is_main)
    emb_key = args.embeddings_key or cfg.get("embeddings_key") or "embeddings"
    use_emb = bool(cfg.get("use_embeddings", True))

    # require_geff=False: the test split has no annotation. load_video already
    # copes -- it attaches links only when a .geff sits beside the stack -- so
    # the videos come back with an empty `links`, which is all inference needs.
    node_files = {p.name: p for p in find_coord_files(args.nodes_folder)}
    paths = sorted(list_datasets(Path(args.data_dir), require_geff=False))
    if args.limit:
        paths = paths[: args.limit]

    videos = []
    for path in paths:
        key = path.name + ".npz" if (path.name + ".npz") in node_files else path.name
        node_path = node_files.get(key)
        if node_path is None:
            if ctx.is_main:
                print(f"!  no detections for {path.name}, skipped")
            continue
        video = load_video(
            path.name, node_path, path,
            use_embeddings=use_emb, embeddings_key=emb_key,
        )
        n_before = len(video.coords)
        video = cap_nodes(video, args.max_nodes)
        if len(video.coords) != n_before and ctx.is_main:
            print(f"   {path.name}: {n_before} -> {len(video.coords)} nodes "
                  f"(cap {args.max_nodes}/frame)")
        videos.append(video)
    if not videos:
        raise FileNotFoundError(
            f"no stack of {args.data_dir} had a matching npz in {args.nodes_folder}"
        )

    dataset = dataset_for(cfg, videos, max_nodes_per_frame=args.max_nodes_per_frame)

    if ctx.is_main:
        print(f"-> checkpoint     : {ckpt_path}")
        print(f"-> nodes_folder   : {args.nodes_folder}")
        print(f"-> embeddings key : {emb_key}  (use_embeddings={use_emb})")
        print(f"-> videos         : {len(videos)}")
        total = sum(len(v.coords) for v in videos)
        per_frame = max((len(r) for v in videos for r in v.rows_by_frame.values()),
                        default=0)
        print(f"-> confidence cap : {args.max_nodes or 'none'} nodes/frame"
              f"   (widest frame after it: {per_frame}, {total} nodes total)")
        print(f"-> dataset cap    : {dataset.max_nodes_per_frame or 'none'}"
              f"   window {dataset.window_size}")
        print(f"-> edge budget    : {edge_predict.EDGE_PAIR_BUDGET:,} pairs/forward")
        print(f"-> threshold      : {args.threshold}")
        print(f"-> writing to     : {out_dir}")
        if ctx.distributed:
            print(f"-> distributed    : {ctx.world_size} processes")
        print()

    mine = list(range(len(videos)))[ctx.rank :: ctx.world_size]
    for i, vi in enumerate(mine):
        video = videos[vi]
        scored = score_video(model, dataset, vi, device, fp16=not args.no_fp16)
        coords, edges = link_video(video, scored, threshold=args.threshold)
        np.savez_compressed(
            out_dir / f"{video.name}.npz",
            coords=np.round(np.asarray(coords, np.float64)).astype(np.int16),
            edges=np.asarray(edges, np.float64).reshape(-1, 4),
            candidates=candidate_array(video, scored),
        )
        print(f"[rank {ctx.rank}] [{i + 1:>3}/{len(mine)}] {video.name}  "
              f"{len(coords)} nodes  {len(edges)} edges", flush=True)


if __name__ == "__main__":
    try:
        main()
    finally:
        cleanup_distributed()

Writing edge_inference.py


In [9]:
cmd = (
    "torchrun --standalone --nproc_per_node=2 edge_inference.py"
    f" --log_folder '{EDGE_W}'"
    " --ckpt 'fold_fullfit/ckpt_final.pt'"
    " --nodes_folder /tmp/preds/"
    f" --data_dir '{DATA_DIR}'"
    " --out_dir /tmp/preds_edge/"
    f" --threshold {EDGE_THRESHOLD}"
    f" --embeddings_key '{EMB_KEY}'"
    f" --max_nodes {MAX_NODES}"
)
print(cmd, "\n")
!{cmd}

torchrun --standalone --nproc_per_node=2 edge_inference.py --log_folder '/kaggle/input/datasets/theoviel/biohub-weights-2/edge_2026-09-09_0/' --ckpt 'fold_fullfit/ckpt_final.pt' --nodes_folder /tmp/preds/ --data_dir '/tmp/data/' --out_dir /tmp/preds_edge/ --threshold 0.45 --embeddings_key 'embeddings_contrastive_2026-09-09_1' --max_nodes 1500 

W0910 11:42:03.314000 164 torch/distributed/run.py:852] 
W0910 11:42:03.314000 164 torch/distributed/run.py:852] *****************************************
W0910 11:42:03.314000 164 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0910 11:42:03.314000 164 torch/distributed/run.py:852] *****************************************
[W910 11:42:03.987153808 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W910 11:42:09.275934266

## 4. Edge post-processing

`src/main_edge_postprocess.py`, on the settings that score **cv 0.8735** over
the 199 out-of-fold stacks of `logs/edge_run/2026-09-13/0/preds_0.45` — the
best row of the shared-configuration table in `Experiments.md`:

    ilp-node-filter(p>=0.4, dis=0.75) -> assign(th=0.4, momentum w=0.02 m=0.7)
        -> recover(gap<=3, r=5) -> divisions(p>=0.3, diverge=2.5, cap=0.00375)
        -> prune(min_track=5) -> linefit(w=0.6, +-3)

Every knob is passed explicitly rather than left to `DEFAULTS`. They happen to
be that block's current values, but these are retuned per checkpoint and do not
carry across them, so relying on whatever `DEFAULTS` the uploaded `src/` holds
would silently score a different chain than the one measured.

Two changes from the previous submission, both worth their cost:

- **The ILP node filter is on.** It was off when it was worth +0.0002; on this
  checkpoint the shared-configuration survey has it in every one of the five
  best runs, at this same 0.75 / 0.40. It costs 9–16 s a stack against ~1 s for
  the rest of the chain, which is why `--jobs` matters more here than it did.
- `div_diverge_um` 4.0 → 2.5 and `div_min_prob` 0.25 → 0.30, from the fold-0
  retune. Together with the ILP these take the old 0.8699 chain to 0.8735.

`--no_score` because the test split ships no ground truth.

In [ ]:
!python src/main_edge_postprocess.py \
    --preds_dir "/tmp/preds_edge/" \
    --out_dir "/tmp/preds_pp/" \
    --geff_dir "/tmp/preds_pp_geff/" \
    --threshold 0.40 \
    --ilp \
    --ilp_threshold 0.40 \
    --ilp_disappearance 0.75 \
    --recover_max_gap 3 \
    --recover_max_residual 5.0 \
    --recover_min_dup 3.0 \
    --min_track 5 \
    --linefit 0.6 \
    --linefit_window 3 \
    --vel_weight 0.02 \
    --vel_momentum 0.7 \
    --div_min_prob 0.30 \
    --div_max_parent_um 11.0 \
    --div_max_existing_child_um 10.0 \
    --div_max_sister_um 14.0 \
    --div_symmetry_tau 0.0 \
    --div_diverge_um 2.5 \
    --div_frame_frac_cap 0.0076 \
    --div_global_frac_cap 0.00375 \
    --no_score \
    --jobs 4

## 5. Division classifier gate  *(USE_DIVISION_CLS)*

The only stage besides the detector that reads pixels. It links nothing: it
scores each fork the chain above proposed — a 32×32 crop over 3 frames centred
on the parent, plus the proposal's own geometry — and deletes the second-child
edge of the ones it disbelieves.

**cv 0.8735 → 0.8791**, entirely in the division term (0.1619 → 0.2150) with
`adj_edge_jaccard` unmoved (0.8574 → 0.8576). Deleting a second child cannot
break a track, so the edge metric has almost nothing to lose; the 121 division
false positives it removes are pure gain, against the 9 true ones it takes
with them.

Why a classifier and not another gate: precision is what binds here, and the
geometry and link-probability features the proposal was *made* with cannot
supply it — every rearrangement of them was measured and failed (see
`postprocessing/README_edge.md`). The image is where the mitotic figure
actually is.

When the flag is off this cell is a passthrough and `GEFF_DIR` stays on the
post-processing output, so the submission below is unchanged.

In [ ]:
# Everything downstream reads GEFF_DIR, so the flag changes one variable rather
# than the submission cell.
GEFF_DIR = "/tmp/preds_pp_geff/"

if USE_DIVISION_CLS:
    cmd = (
        "python src/division_cls/apply_fork_gate.py"
        " --preds_dir /tmp/preds_pp/"
        f" --data_dir '{DATA_DIR}'"
        f" --run_dir '{DIVISION_W}'"
        " --out_dir /tmp/preds_pp_gated/"
        " --geff_dir /tmp/preds_pp_gated_geff/"
        f" --tau {DIVISION_TAU}"
        # "mean" rather than a fold: a test stack is out of fold for both models
        # and belongs to neither, so neither is the right one to pick.
        " --fold mean"
    )
    print(cmd, "\n")
    !{cmd}
    GEFF_DIR = "/tmp/preds_pp_gated_geff/"

print("-> submission reads:", GEFF_DIR)

## Submission

`src/submission.py` again, rather than a copy — the two had already converged
to the same file apart from a warnings filter.

In [ ]:
!python src/submission.py {GEFF_DIR} -o ./submission.csv

In [12]:
sub = pd.read_csv('./submission.csv')
sub.sample(5)

,id,dataset,row_type,node_id,t,z,y,x,source_id,target_id
4949,4949,44b6_18ced818,node,4949,6,7,105,24,-1,-1
169170,169170,44b6_18ced818,edge,-1,-1,-1,-1,-1,78417,93387
20802,20802,44b6_18ced818,node,20802,23,47,94,44,-1,-1
29685,29685,44b6_18ced818,node,29685,33,47,121,205,-1,-1
107699,107699,44b6_18ced818,edge,-1,-1,-1,-1,-1,14508,15428


Done !